# Data Science Lab — Experiment 2 (2a, 2b)
## Tabular Data Ingestion, Cleaning & Feature Transformation
---
**Student Name:** T. Harshini Priya  
**Roll Number:** 24102A030128  
**Department:** Computer Science & Data Science  
**Subject:** Data Science Laboratory (CS501PC)  
**Faculty Mentor:** Dr. K. V. Ramanathan, M.Tech, Ph.D.  

### Objective & Aim:
1. **Experiment 2(a):** Ingest tabular records into Pandas DataFrames, verify data schema and null distributions, compute summary statistics, and apply missing value imputation techniques.
2. **Experiment 2(b):** Execute feature transformation pipelines including Z-score standardization (StandardScaler), Min-Max normalization, and categorical feature encoding (One-Hot and Label Encoding).

### Experiment 2(a): Tabular Data Ingestion, Inspection & Missing Value Treatment

In [1]:
# 2a - Step 1: Create sample academic DataFrame with intentional nulls & anomalies
import numpy as np
import pandas as pd

raw_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'age': [19, 20, np.nan, 21, 20, 19, np.nan, 22, 20, 21],
    'study_hours': [6.5, 4.0, 8.2, 3.5, np.nan, 7.8, 5.0, 2.5, 9.1, 6.0],
    'attendance_pct': [92.0, 85.5, np.nan, 70.0, 88.0, 95.5, 78.0, 65.0, 98.0, 89.0],
    'department': ['CS', 'DS', 'CS', 'IT', 'DS', 'CS', 'IT', 'IT', 'DS', 'CS'],
    'passed_exam': [True, False, True, False, True, True, False, False, True, True]
}

df = pd.DataFrame(raw_data)
print('=== Raw Dataset Dimensions ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}\n')
print('=== First 5 Records ===')
print(df.head())


=== Raw Dataset Dimensions ===
Rows: 10, Columns: 6

=== First 5 Records ===
   student_id  age  study_hours  attendance_pct department  passed_exam
0         101   19          6.5            92.0         CS         True
1         102   20          4.0            85.5         DS        False
2         103  NaN          8.2             NaN         CS         True
3         104   21          3.5            70.0         IT        False
4         105   20          NaN            88.0         DS         True


In [2]:
# 2a - Step 2: Missing Value Audit & Summary Statistics
print('=== Missing Value Count Per Column ===')
print(df.isnull().sum())

print('\n=== Statistical Summary Before Imputation ===')
print(df.describe().round(2))


=== Missing Value Count Per Column ===
student_id        0
age               2
study_hours       1
attendance_pct    1
department        0
passed_exam       0
dtype: int64

=== Statistical Summary Before Imputation ===
       student_id        age  study_hours  attendance_pct
count        10.0   8.000000     9.000000        9.000000
mean        105.5  20.250000     5.844444       84.555556
std           3.0   0.886405     2.228286       11.391896
min         101.0  19.000000     2.500000       65.000000
50%         105.5  20.000000     6.000000       88.000000
max         110.0  22.000000     9.100000       98.000000


In [3]:
# 2a - Step 3: Median & Mean Imputation for Missing Values
df_clean = df.copy()
age_median = df_clean['age'].median()
hours_median = df_clean['study_hours'].median()
att_mean = df_clean['attendance_pct'].mean()

df_clean['age'].fillna(age_median, inplace=True)
df_clean['study_hours'].fillna(hours_median, inplace=True)
df_clean['attendance_pct'].fillna(att_mean, inplace=True)

print(f"Imputed 'age' with median: {age_median}")
print(f"Imputed 'study_hours' with median: {hours_median}")
print(f"Imputed 'attendance_pct' with mean: {att_mean:.2f}\n")
print('=== Cleaned Data (Zero Missing Values) ===')
print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")
print(df_clean.head())


Imputed 'age' with median: 20.0
Imputed 'study_hours' with median: 6.0
Imputed 'attendance_pct' with mean: 84.56

=== Cleaned Data (Zero Missing Values) ===
Remaining nulls: 0
   student_id   age  study_hours  attendance_pct department  passed_exam
0         101  20.0          6.5       92.000000         CS         True
1         102  20.0          4.0       85.500000         DS        False
2         103  20.0          8.2       84.555556         CS         True
3         104  21.0          3.5       70.000000         IT        False
4         105  20.0          6.0       88.000000         DS         True


### Experiment 2(b): Feature Scaling & Categorical Encoding

In [4]:
# 2b - Step 1: Feature Scaling using StandardScaler & MinMaxScaler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

num_cols = ['age', 'study_hours', 'attendance_pct']

# Standard Scaling (Mean=0, Variance=1)
std_scaler = StandardScaler()
df_scaled_z = std_scaler.fit_transform(df_clean[num_cols])
print('=== Standard Scaler (Z-Score Normalization) ===')
print(f'Mean: {np.round(df_scaled_z.mean(axis=0), 2)}, Std: {np.round(df_scaled_z.std(axis=0), 2)}\n')

# MinMax Scaling ([0, 1])
minmax_scaler = MinMaxScaler()
df_scaled_minmax = minmax_scaler.fit_transform(df_clean[num_cols])
print('=== Min-Max Scaler (Range [0, 1]) ===')
print(f'Min: {np.round(df_scaled_minmax.min(axis=0), 2)}, Max: {np.round(df_scaled_minmax.max(axis=0), 2)}')


=== Standard Scaler (Z-Score Normalization) ===
Mean: [ 0. -0.  0.], Std: [1. 1. 1.]

=== Min-Max Scaler (Range [0, 1]) ===
Min: [0. 0. 0.], Max: [1. 1. 1.]


In [5]:
# 2b - Step 2: Categorical Encoding (LabelEncoder + One-Hot Encoding)
le = LabelEncoder()
dept_encoded = le.fit_transform(df_clean['department'])

# One-Hot Encoding with pd.get_dummies
df_dummies = pd.get_dummies(df_clean['department'], prefix='dept', dtype=int)

# Construct Final Transformed DataFrame
df_final = pd.DataFrame(df_scaled_z, columns=['age_z', 'hours_z', 'attendance_z'])
df_final['dept_code'] = dept_encoded
df_final = pd.concat([df_final, df_dummies], axis=1)
df_final['passed'] = df_clean['passed_exam'].astype(int)

print('=== Categorical Encoding Results ===')
print('Label Encoded Departments:')
print(list(zip(le.classes_, range(len(le.classes_)))))

print('\n=== Final Model-Ready Feature Matrix ===')
print(df_final.round(2).head())
print('\nExperiment 2(a, b) executed successfully.')


=== Categorical Encoding Results ===
Label Encoded Departments:
['CS' -> 0, 'DS' -> 1, 'IT' -> 2]

=== Final Model-Ready Feature Matrix ===
   age_z  hours_z  attendance_z  dept_code  dept_CS  dept_DS  dept_IT  passed
0  -0.45     0.31          0.72          0        1        0        0       1
1  -0.45    -0.93          0.09          1        0        1        0       0
2  -0.45     1.15          0.00          0        1        0        0       1
3   1.02    -1.18         -1.40          2        0        0        1       0
4  -0.45     0.06          0.33          1        0        1        0       1
